In [ ]:
!pip install -q langchain==0.3.26 langchain-community==0.3.27 langchain-groq langchain-text-splitters chromadb sentence-transformers pypdf

In [ ]:
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
pages=PyPDFLoader(pdf_name).load()
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(pages)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db=Chroma.from_documents(chunks,embeddings)

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
llm=ChatGroq(api_key="YOUR_GROQ_API_KEY",model="llama-3.1-8b-instant",temperature=0)
qa=RetrievalQA.from_chain_type(llm=llm,retriever=db.as_retriever())
result=qa.invoke({"query":"What is this document about?"})
print(result["result"])